# Notebook 2 - $\color{green}\text{Watershed Splitter}$

In [1]:
# Packages (geostack)
import rasterio as rio, shapely, json, requests, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, xdem, rioxarray as rxr

from pathlib import Path
from tqdm import tqdm
from shapely.geometry import shape
from rasterio.mask import mask
from rasterio.crs import CRS
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# User defined parameters. Run this cell each time you update one of the enclosed variables.
BOUNDS = [1,2,3,4] # In decimal coordinate form.  Format: [N, E, S, W].

PROJ_TITLE = 'example_directory' # Create a succinct name with no spaces or leading digits to represent your project file for future exports.

PATH = Path.cwd() # Gets the user's current working directory to avoid the use of absolute paths.


***
The below cell will split the main DEM into individual watersheds at the user's chosen scale.  The default is HUC-10.

In [13]:
wshed_path = Path(PATH) / PROJ_TITLE / 'wshed_dems'
print(wshed_path)

d:\thesis_data_dict\notebooks_blank\example_directory\wshed_dems


In [4]:
# Wshed Clipper
def wshed_splitter(mode, proj_title = PROJ_TITLE, bounds = BOUNDS, path = PATH, size = 'HUC10'):
    """
    Creates seperate DEMs for each watershed in the user's selected bounds.  Places in their own folder within user's project directory.

    args: proj_title (str), bounds (list), path (current working directory), size (str) HUC10, 12, or 14.

    returns: None.
    """
    print('Loading data...')

    size_dict = {'HUC10': 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/5/query?',  
    'HUC12': 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/6/query?',
    'HUC14': 'https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/7/query?'}
    url = size_dict[size]

    user_AOI = f'{bounds[3]}, {bounds[2]}, {bounds[1]}, {bounds[0]}'

    #the parameters here will query the map server for the appropriate HU boundary
    params = dict(geometry=user_AOI,geometryType='esriGeometryEnvelope',inSR='4326',
                spatialRel='esriSpatialRelIntersects',f='geojson')

    #Execute REST API call.  
    try:
        r = requests.get(url,params=params)
    except:
        print('Error with Service Call. This could mean that there is no hydrologic unit polygon where you selected.')

    # load API JSON output into a variable
    wbd_geojson = json.loads(r.content)

    features = wbd_geojson['features']
    geometries = [shape(feature['geometry']) for feature in features]
    properties = [feature['properties']['name'] for feature in features]

    w_gdf_temp = pd.DataFrame({'Name': properties, 'geometry0' : geometries})
    gdf = gpd.GeoDataFrame(w_gdf_temp, geometry=w_gdf_temp['geometry0'], crs="EPSG:4326")
    
    clipped = gpd.GeoDataFrame(gdf.cx[bounds[3]:bounds[1], bounds[2]:bounds[0]]) # Clip it to the user's bounds.

    # Now that we have a GDF (GeoDataFrame) with the user's chosen watersheds and the larger dem, lets export DEMs of each watershed.
    try:
        wshed_path = Path(path) / proj_title / 'wshed_dems'
        wshed_path.mkdir() # Makes a directory to store these new dems.
        print(f'Directory "{wshed_path}" created successfully.') # Tries to make the directory for project files.
    except FileExistsError:
        print(f'Directory "{wshed_path}" already exists.') # If it doesnt work, it'll tell you!
    except Exception as e:
        print(f'An error occurred: {e}')
    
    # We have our directory, lets populate it.
    print('Writing new DEMs...')
    with rio.open(Path(path) / proj_title / f"gtiff_{str(BOUNDS).replace(' ', '').replace(',','_').replace('[', '').replace(']','')}.tiff") as src: # Loads in our larger df.
        total_tasks=len(clipped)
        with tqdm(total=total_tasks, desc='Progress') as pbar: # This is for the fancy progress bar.
            for name_orig in clipped['Name'].unique():
                name = name_orig.replace(' ', '_')
                if mode == 'shape':
                    geom = (clipped.loc[clipped['Name']==name_orig].geometry)
                if mode == 'bounds':
                    geom = (clipped.loc[clipped['Name']==name_orig].geometry)
                    geom = shapely.geometry.box(float(geom.bounds['minx'].iloc[0]), float(geom.bounds['miny'].iloc[0]), float(geom.bounds['maxx'].iloc[0]), float(geom.bounds['maxy'].iloc[0]))
                else:
                    print('Please select mode ("shape" or "bounds")')

                out_image, out_transform = mask(src, [geom], crop=True) # Clips the DEM to our watershed's shape.  Sets nodata value to LSDTopyTools standard.

                # Below is metadata stuff, dont stress about this.
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                    })

                out_path = Path(path) / proj_title / 'wshed_dems' / f'{name}.tiff' # Create a pathname.
                with rio.open(out_path, "w", **out_meta) as file:
                    file.write(out_image) # Write to file.
                pbar.update(1)
    

# Call our function, let it do the work.           
wshed_splitter(mode = 'bounds')

Loading data...
Directory "d:\thesis_data_dict\notebooks_blank\example_directory\wshed_dems" created successfully.
Writing new DEMs...


Progress: 100%|██████████| 14/14 [00:00<00:00, 31.34it/s]


Now that we have our DEMs we will do some relief mapping and reprojecting for future knickpoint analysis.

In [5]:
# Relief
def map_relief(dem_dir=Path(PATH) / PROJ_TITLE / 'wshed_dems'):
    """
    Takes DEMs of watersheds and produces TRI (Terrain ruggedness index, an SLRM) for each DEM.  Saves to own folder.

    args: dem_dir (path to wshed DEMs folder).

    returns: None.
    """
    # Glob a list of paths to the DEMs.
    path_list = list(dem_dir.glob('*.tiff'))
 
    # Makes a directory to store relief maps in.
    try:
        tri_path = Path(PATH) / PROJ_TITLE / 'tri_pngs'
        tri_path.mkdir()
        print(f'Directory "{tri_path}" made sucessfully.')
    except FileExistsError:
        print(f'Directory "{tri_path}" Exists')
        pass
    except Exception as e:
        print(f'An error occured: {e}')
    print('Saving figures...')

    # Progress bar.
    with tqdm(total=len(path_list), desc='Progress') as pbar:
        for path in path_list:
            # For each path, create a TRI (terrain ruggedness index, simple local relief model).  Then map this TRI model.
            tri = xdem.DEM(path).terrain_ruggedness_index()
            plt.ioff()
            fig, ax = plt.subplots(dpi=250)
            _ = tri.plot(cmap='magma', ax=ax, cbar_title='Relief (m)')
            name = path.name[:-5]
            name_no_und = name.replace('_',' ')
            ax.set_title(f'{name_no_und} Terrain Ruggedness Index')
            ax.set_ylabel('Latitude (°)')
            ax.set_xlabel('Longitude (°)')
            fig.tight_layout()
            save_path = Path(str(tri_path)) / f'{name}.png'
            plt.savefig(save_path)
            plt.close()
            pbar.update(1)

map_relief()

Directory "d:\thesis_data_dict\notebooks_blank\example_directory\tri_pngs" made sucessfully.
Saving figures...


Progress: 100%|██████████| 14/14 [00:09<00:00,  1.47it/s]


In [ ]:
# Reprojecting to northing, easting. (needed for KSN alalysis)

# Define our function.
def reproject(dem_path):
    dem_name = dem_path.name[:-5] # Get succinct DEM name.
    print(f'Reprojecting {dem_name}')
    crs = CRS.from_epsg(2283) # Set the new CRS.
    dem = rxr.open_rasterio(dem_path) # Open the DEM.
    reprojected_dem = dem.rio.reproject(dst_crs=crs) # Reproject.
    # Save to new directory.
    reprojected_dem.rio.to_raster(Path(PATH) / PROJ_TITLE / 'reproj' / f'{dem_name}_reproj.tiff')

# Create directory.
try:
    reproj_dir = Path(PATH) / PROJ_TITLE / 'reproj'
    reproj_dir.mkdir()
except FileExistsError:
    print('Directory exists')

# Glob a list of paths.
dem_dir=Path(PATH) / PROJ_TITLE / 'wshed_dems'
paths=list(dem_dir.glob('*.tiff'))

# Run function in parallel to save time.
with ThreadPoolExecutor() as executor:
    executor.map(reproject, paths)

Reprojecting Big_Sandy_Creek
Reprojecting Coxes_Creek
Reprojecting Deep_Creek
Reprojecting Georges_Creek
Reprojecting Headwaters_Youghiogheny_River
Reprojecting Indian_Creek
Reprojecting Laurel_Hill_Creek
Reprojecting Lower_Casselman_River
Reprojecting Middle_Youghiogheny_River
Reprojecting New_Creek-North_Branch_Potomac_River
Reprojecting Savage_River
Reprojecting Stonycreek_River
Reprojecting Upper_Casselman_River
Reprojecting Upper_Youghiogheny_River
